# Visualize Best-Checkpoint Live Vehicle Replay

Use this notebook to restore one RLlib run, auto-select a retained best-validation checkpoint, replay a TraCI evaluation episode, and render the live vehicle movement as a GIF.

## What this notebook does

1. Locates the repo root and Python environment.
2. Points at one Hydra RLlib `RUN_DIR`.
3. Restores the best retained validation checkpoint.
4. Replays one evaluation episode on TraCI.
5. Saves `trip_trace.json`, `trip_animation.gif`, and `trip_animation_metadata.json` under `experiments/artifacts/live_trip_viz/`.

This notebook is a thin front-end over `sumo_rl.experiments.live_trip_visualization`.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

from IPython.display import Image as IPyImage, display

try:
    import pandas as pd
except ImportError:
    pd = None


def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "sumo_rl").exists() and (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Could not locate the repo root from the current working directory.")


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from sumo_rl.experiments.live_trip_visualization import run_best_checkpoint_trip_visualization

print(f"Repo root: {ROOT}")
print(f"pandas available: {pd is not None}")

In [ ]:
RUN_DIR = ROOT / "outputs" / "replace_me_with_hydra_run_dir"
BEST_INDEX = 0
WIDTH = 1200
FPS = 12
FRAME_COUNT = 180
MAX_RENDER_VEHICLES = 2000
OUTPUT_DIR = None

RUN_DIR

In [ ]:
RUN_DIR = Path(RUN_DIR).expanduser().resolve()
if not RUN_DIR.exists():
    raise FileNotFoundError(f"RUN_DIR does not exist: {RUN_DIR}")

RESOLVED_OUTPUT_DIR = (
    Path(OUTPUT_DIR).expanduser().resolve()
    if OUTPUT_DIR is not None
    else ROOT / "experiments" / "artifacts" / "live_trip_viz" / RUN_DIR.name
)

print(f"Run dir: {RUN_DIR}")
print(f"Output dir: {RESOLVED_OUTPUT_DIR}")

In [ ]:
paths = run_best_checkpoint_trip_visualization(
    RUN_DIR,
    RESOLVED_OUTPUT_DIR,
    best_index=int(BEST_INDEX),
    width=int(WIDTH),
    fps=int(FPS),
    frame_count=int(FRAME_COUNT),
    max_render_vehicles=int(MAX_RENDER_VEHICLES),
)

for label, path in paths.items():
    print(f"{label}: {path}")

## Preview the GIF

In [ ]:
display(IPyImage(filename=str(paths["animation"])))

## Load saved metadata and trace summary

In [ ]:
metadata = json.loads(Path(paths["metadata"]).read_text(encoding="utf-8"))
trace = json.loads(Path(paths["trace"]).read_text(encoding="utf-8"))

print(json.dumps(metadata, indent=2)[:4000])
print(f"Total frames in trace: {len(trace.get('frames', []))}")

In [ ]:
summary_row = {
    "run_dir": metadata.get("run_dir"),
    "checkpoint_rank": metadata.get("checkpoint_rank"),
    "metric_name": metadata.get("metric_name"),
    "metric_value": metadata.get("metric_value"),
    "seed": metadata.get("seed"),
    "trace_frames": metadata.get("trace_frames"),
    "max_live_vehicles": metadata.get("max_live_vehicles"),
    "tls_count": metadata.get("tls_count"),
}

if pd is not None:
    display(pd.DataFrame([summary_row]))
else:
    print(summary_row)

In [ ]:
for path in sorted(Path(RESOLVED_OUTPUT_DIR).rglob("*")):
    print(path.relative_to(RESOLVED_OUTPUT_DIR))